# Laboratório 09: A Engenharia do Tempo (Datetime)
**Disciplina:** Extração e Preparação de Dados (IBM8915)
**Objetivo:** Aprender a converter strings de data para o tipo `datetime64[ns]` e utilizar o acessor `.dt` do Pandas para extrair sazonalidade (mês, dia da semana) e calcular deltas de tempo.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Parte 1: Exemplo Guiado - Dissecando o Tempo
O texto '17/03/2026' é inútil para a matemática. Precisamos convertê-lo e dissecá-lo. O Pandas possui o método `to_datetime` e o acessor `.dt` para isso.

In [2]:
df_tempo = pd.DataFrame({
    'Data_Assinatura': ['15/01/2024', '20/02/2024', '05/03/2024'],
    'Data_Cancelamento': ['20/01/2024', '25/03/2024', '15/03/2024']
})

# 1. Conversão (Atenção ao dayfirst=True para o padrão BR: Dia/Mês/Ano)
df_tempo['Data_Assinatura'] = pd.to_datetime(df_tempo['Data_Assinatura'], dayfirst=True)
df_tempo['Data_Cancelamento'] = pd.to_datetime(df_tempo['Data_Cancelamento'], dayfirst=True)

# 2. Extração de Sazonalidade com .dt
df_tempo['Mes_Assinatura'] = df_tempo['Data_Assinatura'].dt.month
df_tempo['Dia_Semana'] = df_tempo['Data_Assinatura'].dt.dayofweek # 0=Segunda, 6=Domingo

# 3. Matemática Temporal (Deltas)
delta_tempo = df_tempo['Data_Cancelamento'] - df_tempo['Data_Assinatura']
df_tempo['Dias_Ativo'] = delta_tempo.dt.days # Extraindo o número inteiro de dias

display(df_tempo)

,Data_Assinatura,Data_Cancelamento,Mes_Assinatura,Dia_Semana,Dias_Ativo
0,2024-01-15,2024-01-20,1,0,5
1,2024-02-20,2024-03-25,2,1,34
2,2024-03-05,2024-03-15,3,1,10


## Parte 2: Exercício Prático e Desafio
A equipe de marketing quer entender o comportamento de compra aos finais de semana. 
**Sua Tarefa:**
1. Converta a coluna `Data_Compra` do formato texto para `datetime64`.
2. Extraia o `Ano`, `Mes` e `Dia_Semana` para novas colunas.
3. **O Desafio da Flag:** Crie uma coluna booleana (0 ou 1) chamada `Is_Weekend`. Considere que os dias 5 (Sábado) e 6 (Domingo) formam o fim de semana. Dica: use `(condicao).astype(int)`.

In [3]:
np.random.seed(42)
datas_sujas = [f"{np.random.randint(1, 28)}/0{np.random.randint(1, 9)}/2025" for _ in range(10)]
df_vendas = pd.DataFrame({'ID_Venda': range(1, 11), 'Data_Compra': datas_sujas})

In [4]:
df_vendas

,ID_Venda,Data_Compra
0,1,7/04/2025
1,2,15/03/2025
2,3,8/05/2025
3,4,21/07/2025
4,5,26/03/2025
5,6,23/03/2025
6,7,11/08/2025
7,8,21/04/2025
8,9,8/08/2025
9,10,3/06/2025


In [6]:
# Passo 1: Use pd.to_datetime com dayfirst=True
df_vendas['Data_Compra'] = pd.to_datetime(df_vendas['Data_Compra'], dayfirst=True)

# Passo 2: Extraia Ano, Mês e Dia_Semana usando .dt
df_vendas['mês'] = df_vendas['Data_Compra'].dt.month
df_vendas['ano'] = df_vendas['Data_Compra'].dt.year

# Passo 3: Crie a flag Is_Weekend (1 para Fim de Semana, 0 para Dia Útil)
df_vendas['dia da semana'] = df_vendas['Data_Compra'].dt.dayofweek # 0=Segunda, 6=Domingo

df_vendas['Fim de semana'] = (df_vendas['dia da semana'] >4).astype(int)

In [7]:
df_vendas

,ID_Venda,Data_Compra,mês,ano,dia da semana,Fim de semana
0,1,2025-04-07,4,2025,0,0
1,2,2025-03-15,3,2025,5,1
2,3,2025-05-08,5,2025,3,0
3,4,2025-07-21,7,2025,0,0
4,5,2025-03-26,3,2025,2,0
5,6,2025-03-23,3,2025,6,1
6,7,2025-08-11,8,2025,0,0
7,8,2025-04-21,4,2025,0,0
8,9,2025-08-08,8,2025,4,0
9,10,2025-06-03,6,2025,1,0
